In [11]:
%load_ext autoreload
%autoreload 2

from pklib.indicators import *
from pklib.utilities import *
from pklib.pkindicators import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
# import pkindicators  # Assuming your C module is compiled and installed as pkindicators

# Generate sample data (you should replace this with real OHLCV data)
# dates = pd.date_range('2022-01-01', periods=1000, freq='D')
# prices = np.random.randn(1000).cumsum() + 100  # Simulated price data
# data = pd.DataFrame(prices, columns=['close'], index=dates)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:

import sysconfig
cmodule = 'pkindicators'
f'clear & rm {cmodule}.so & gcc -shared -o {cmodule}.so -fPIC {cmodule}.c -I{sysconfig.get_path("include")} -I{np.get_include()}'

'clear & rm pkindicators.so & gcc -shared -o pkindicators.so -fPIC pkindicators.c -I/home/mu6mula/miniconda3/envs/py310/include/python3.10 -I/home/mu6mula/miniconda3/envs/py310/lib/python3.10/site-packages/numpy/core/include'

In [ ]:
# fib_levels = np.array([-1.0, -0.786, -0.618, -0.5, -0.382, -0.236, 0.0, 0.236, 0.382, 0.5, 0.618, 0.786, 1.0, 1.236, 1.5, 1.618, 1.786, 2.0, 2.236, 2.382, 2.5, 2.628, 2.786, 3, 3.382, 3.618, 4, 5])
fib_levels = np.array([-1.0, -0.786, -0.618, -0.5, -0.382, -0.236, 0.0, 0.236, 0.382, 0.5, 0.618, 0.786, 1.0, 1.236, 1.5, 1.618, 1.786, 2.0])

fib_columns = [f'fib({fib})' for fib in fib_levels]
# fib_columns 

def calculate_fib_levels(data, epsilon, fib_levels=fib_levels):
    
    # Call the calculate_zigzag function from the C module
    high_low_markers, turning_points = calculate_zigzag(data['close'].values, data['close'].values, epsilon=epsilon)
    

    # Store the results back into the DataFrame for easier plotting
    data['HighLowMarkers'] = high_low_markers
    data['TurningPoints'] = turning_points

    # Get the indices of highs and lows
    highs_idx = data.index[data['HighLowMarkers'] == 1]
    lows_idx = data.index[data['HighLowMarkers'] == -1]

    running_highs = (np.where(high_low_markers == 1, 1, np.nan) * data['close']).ffill()
    running_lows = (np.where(high_low_markers == -1, 1, np.nan) * data['close']).ffill()

    running_highs_idx = pd.Series(np.where(high_low_markers == 1, 1, np.nan) * np.arange(len(data)), index=data.index).ffill()
    running_lows_idx = pd.Series(np.where(high_low_markers == -1, 1, np.nan) * np.arange(len(data)), index=data.index).ffill()


    directions = ((running_highs_idx > running_lows_idx) * 2 - 1)
    # Combine the indices and sort them
    turning_points_idx = sorted(highs_idx.union(lows_idx))

    # Calculate Fibonacci levels (assuming calculate_fib_levels and fib_columns are defined)
    # fibs = calculate_fib_levels(running_lows, running_highs, ((running_highs_idx > running_lows_idx) * 2 - 1))
    
    # Ensure highs, lows, and directions are numpy arrays
    # highs = np.asarray(highs)
    # lows = np.asarray(lows)
    # directions = np.asarray(directions)

    # Flip highs and lows based on direction (-1 means flip)
    adjusted_highs = np.where(directions == 1, running_highs, running_lows)
    adjusted_lows = np.where(directions == 1, running_lows, running_highs)

    # Calculate the difference between adjusted high and low
    diff = adjusted_highs - adjusted_lows

    # Calculate the Fibonacci levels by applying the levels to the differences
    fib_matrix = np.outer(diff, fib_levels)
    
    # Calculate the actual levels by adding them to the low (base) level
    fib_levels_array = adjusted_lows[:, np.newaxis] + fib_matrix
    
    df_fibs = pd.DataFrame(fib_levels_array, columns=fib_columns, index=data.index)

    return fib_levels_array, df_fibs, (running_highs_idx, running_lows_idx), (running_highs, running_lows), directions, high_low_markers, turning_points, turning_points_idx


In [14]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.dates as mdates
from mplfinance.original_flavor import candlestick_ohlc

def plot_highs_lows_with_fibs(data, epsilon, window=None, candlestick_ohlc_args={}):
    """
    Plot candlesticks with highs and lows markers, and Fibonacci levels.

    Parameters:
    - data: DataFrame containing OHLCV data with columns: 'open', 'high', 'low', 'close', etc.
    - epsilon: Epsilon value used for the ZigZag calculation.
    - window: Optional tuple (start, end) to define the plotting window.
    """

    fib_levels_array, df_fibs, (running_highs_idx, running_lows_idx), (running_highs, running_lows), directions, high_low_markers, turning_points, turning_points_idx = calculate_fib_levels(data, epsilon)

    # If a window is specified, slice the data
    if window is not None:
        start, end = window
        data_to_plot = data.iloc[start:end].copy()
        df_fibs_to_plot = df_fibs.iloc[start:end].copy()
    else:
        data_to_plot = data.copy()
        df_fibs_to_plot = df_fibs.copy()

    # Create the figure and axes
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(20, 10), height_ratios=[2, 1])

    # ---------------------------
    # Candlestick Plotting
    # ---------------------------
    # Convert datetime index to Matplotlib's float dates
    data_to_plot['date_num'] = mdates.date2num(data_to_plot.index)
    # Prepare the OHLC data in the format: (date, open, high, low, close)
    ohlc = data_to_plot[['date_num', 'open', 'high', 'low', 'close']].values
    # Plot the candlesticks
    candlestick_ohlc(ax1, ohlc, colorup='green', colordown='red', alpha=0.8, **candlestick_ohlc_args)

    # ---------------------------
    # Overlay Additional Markers & Lines
    # ---------------------------
    # Plot highs and lows markers (convert datetime index to numeric)
    highs = data_to_plot[data_to_plot['HighLowMarkers'] == 1]
    lows = data_to_plot[data_to_plot['HighLowMarkers'] == -1]
    # ax1.scatter(mdates.date2num(highs.index), highs['close'],
    #             color='green', marker='^', s=100, label='Highs')
    # ax1.scatter(mdates.date2num(lows.index), lows['close'],
    #             color='red', marker='v', s=100, label='Lows')

    # Plot the ZigZag line connecting turning points
    turning_dates = mdates.date2num(data.loc[turning_points_idx].index)
    ax1.plot(turning_dates, data.loc[turning_points_idx, 'close'],
             color='purple', label='ZigZag Line', lw=1.5)

    # Plot the Fibonacci levels
    # Convert the index of df_fibs_to_plot to numeric dates
    df_fibs_to_plot = df_fibs_to_plot.copy()
    df_fibs_to_plot.index = mdates.date2num(df_fibs_to_plot.index)
    df_fibs_to_plot.plot(ax=ax1, linestyle='--', lw=1)

    # Set y-axis limits based on the close price
    ax1.set_ylim(data_to_plot['close'].min(), data_to_plot['close'].max())

    # Format the x-axis to show dates
    ax1.xaxis_date()
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    ax1.set_title('Candlestick Chart with Highs, Lows, and Fibonacci Levels')
    ax1.legend()

    # Optionally add your additional plot to ax2 here
    # ax2.plot(...)

    plt.tight_layout()
    plt.show()


In [15]:
# data

In [16]:
exchange,base,quote,timeframe = 'binance','ETH', 'USDT', '4h'
data = load_candles(exchange,base,quote,timeframe).iloc[-200:].apply(np.log)
plot_highs_lows_with_fibs(data, epsilon=0.1, candlestick_ohlc_args={'width': .5 / np.log(len(data))})

UnboundLocalError: local variable 'running_highs_idx' referenced before assignment